# 5장 2강 · VLM API와 이미지 정보 추출

실습파일

상품 카탈로그의 우산 사진을 한 줄씩 등록하려고 합니다. 사진에서 알 수 있는 정보는 채우고, 흑백 사진에서 확인할 수 없는 실제 색은 비워 두어야 합니다. 사진을 요청에 담는 단계부터 표의 각 값에 근거가 있는지 검수하는 단계까지 진행합니다.

## 실습 목표와 실행 안내

| 구간 | 해결할 문제 | 제출할 근거 |
|---|---|---|
| 필수 1 | 모델이 받은 사진이 맞는가? 색을 지우면 답이 달라지는가? | Base64 왕복 검사, 실제 응답 원문, 동일 사진 전후 비교표 |
| 필수 2 | 자유로운 답을 일정한 필드로 수집하고 실패 사유를 남길 수 있는가? | 스키마 검사, 첫 응답과 재요청 기록, 4장 결과표와 항목별 일치 수 |
| 과제 1 | 색상 보류, 키 누락, 내용 오류, 요청 실패를 어떻게 구분할까? | 사진 근거, 사례 A~D 판정, 실제 모델 결과의 수정 전후 비교 |

**준비물**  기존 강의용 `coco_sample.zip`, `coco_dataset.zip`, 또는 압축을 푼 `coco_dataset` 폴더를 노트북 옆에 두세요. 경로가 다르면 공통 준비 ①의 `files`를 수정합니다. 원본은 COCO 9448·40083·29984·84477 네 장입니다. 비교 실험에서 컬러 원본 세 장으로 흑백 입력도 만듭니다. 정답 파일은 저장된 실행값과 그림을 바로 볼 수 있습니다.

| 실행 방식 | 설정과 필요한 준비 |
|---|---|
| 기본 · 실제 로컬 추론 | `USE_OPENAI=False`. OpenAI Python SDK가 내 컴퓨터의 HTTP 서버로 요청을 보내고, SmolVLM-256M-Instruct가 요청에 든 사진과 질문으로 답을 생성합니다. API 키 없이 CPU에서 실행할 수 있습니다. 첫 실행은 모델 가중치 다운로드를 위해 인터넷 연결이 필요합니다. |
| 선택 · OpenAI 원격 API | `USE_OPENAI=True`. 노트북 옆 `.env`에 `OPENAI_API_KEY`를 설정하고 공통 준비부터 실행합니다. 이미지가 외부 API에 전송되며 사용료가 발생할 수 있습니다. 예제 모델은 `gpt-4.1-mini`입니다. 정답에 적힌 수치는 기본 모드의 결과입니다. |

**검증 환경**  Python 3.12, PyTorch 2.6.0 CPU, torchvision 0.21.0, transformers 4.53.3, openai 2.30.0, Pydantic 2.13.4입니다. 새 환경이면 Jupyter에서 아래 명령을 실행한 뒤 커널을 재시작하세요.

```python
%pip install torch==2.6.0 torchvision==0.21.0 transformers==4.53.3 openai==2.30.0 "pydantic>=2,<3" python-dotenv httpx pandas pillow
```

공통 준비 ①~③을 먼저 실행하세요. 각 구간은 공통 준비 직후부터 독립적으로 시작할 수 있고, 구간 안에서는 문항을 번호순으로 풉니다. 사진을 하나의 질문과 결합하는 모델을 VLM(시각·언어 모델)이라고 합니다. 이 실습은 사전학습 가중치를 고정한 추론 실험입니다. 질문과 추출 필드값은 영어를 사용하고 해석은 한국어로 작성합니다.

### 공통 준비 ① · 원본과 참조 자료 열기

`objects.csv`에는 사람이 표시한 물체의 종류와 위치가 있습니다. 한 사진에 객체가 여러 개면 같은 image_id가 반복되므로, 경로는 ID마다 하나만 남깁니다. `jpeg(id)`는 파일 바이트를 그대로 읽고, `rgb(raw)`는 픽셀 검사용 이미지로 엽니다. 둘은 쓰임새가 다릅니다.

`mono_jpeg`는 밝기 한 채널로 변환한 뒤 새 JPEG 바이트를 만듭니다. `chroma`는 디코딩한 전체 사진에서 R–G, G–B, B–R 차이의 절댓값 중 가장 큰 값을 구합니다. 0이면 모든 픽셀에서 세 채널 값이 같습니다. 값이 양수면 사진 어딘가에 채널 차이가 있다는 뜻이므로 우산 자체의 색은 원본을 보면서 판독해야 합니다.

In [1]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import base64, hashlib, json, math, re
import pandas as pd
from PIL import Image, ImageDraw, ImageOps, ImageChops
from IPython.display import display
from pydantic import BaseModel, ConfigDict, ValidationError

files = [Path("coco_dataset.zip")]
DATA = next((p for p in files if p.exists()),None)
if DATA is None:
    raise FileNotFoundError("강의용 COCO ZIP 또는 coco_dataset 폴더 경로를 files에 지정하세요.")
if DATA.is_dir():
    root = next(DATA.rglob("objects.csv")).parent
    def read_file(name):
        return (root/name).read_bytes()
else:
    with ZipFile(DATA) as z:
        prefix = next(n for n in z.namelist() if n.split("/")[-1]=="objects.csv")[:-len("objects.csv")]
    def read_file(name):
        with ZipFile(DATA) as z:
            return z.read(prefix+name)
objects = pd.read_csv(BytesIO(read_file("objects.csv")))
ids = [9448,40083,29984,84477]
paths = objects.drop_duplicates("image_id").set_index("image_id").image_path
def jpeg(key):
    return read_file(paths.at[key])
def rgb(raw):
    with Image.open(BytesIO(raw)) as im:
        return im.convert("RGB")
def mono_jpeg(raw):
    buf = BytesIO()
    ImageOps.grayscale(rgb(raw)).save(buf,format="JPEG",quality=95)
    return buf.getvalue()
def chroma(raw):
    r,g,b = rgb(raw).split()
    return max(ImageChops.difference(a,c).getextrema()[1] for a,c in [(r,g),(g,b),(b,r)])
def show(items):
    cols = min(3,len(items))
    sheet = Image.new("RGB",(cols*225, math.ceil(len(items)/cols)*186),"white")
    pen = ImageDraw.Draw(sheet)
    for i,(label,raw) in enumerate(items):
        im = ImageOps.contain(rgb(raw),(220,156))
        x,y = i%cols*225,i//cols*186
        sheet.paste(im,(x+(220-im.width)//2,y))
        pen.text((x+2,y+159),str(label),fill="black")
    display(sheet)
print("대상 사진",[(i,rgb(jpeg(i)).size) for i in ids])

대상 사진 [(9448, (551, 640)), (40083, (500, 333)), (29984, (640, 492)), (84477, (640, 427))]


### 공통 준비 ② · 요청이 실제 모델까지 가는 과정

제공 코드입니다. 기본 모드에서는 서버를 `127.0.0.1`에 열고, 커널이 정한 포트 번호를 SDK에 알려 줍니다. ① SDK가 텍스트·이미지 두 항목을 보냅니다. ② 서버가 Base64를 풀고 RGB 사진을 엽니다. ③ 프로세서가 사진과 질문을 모델용 픽셀·토큰으로 바꿉니다. 긴 변 512 설정으로 한 화면을 처리하고, 생성 시 샘플링을 끕니다. ④ 생성한 답만 잘라 HTTP 응답으로 보냅니다. 서버는 파일 ID별 정답을 가지고 있지 않습니다.

로컬 서버는 이 수업에 필요한 요청 형식만 지원합니다. 구조화 요청에서는 스키마를 생성 지시에 포함하지만 JSON 문법을 강제하는 디코더는 사용하지 않습니다. 새 토큰 80개 제한을 만나면 `incomplete`로 표시합니다. 따라서 네트워크 요청 성공, 생성 완료, 구조 검사 통과가 서로 다르게 나타날 수 있습니다. 커널을 종료하면 로컬 서버도 종료됩니다.

In [2]:
import httpx, openai, threading, time, uuid
from http.server import BaseHTTPRequestHandler, HTTPServer
from openai import OpenAI
from dotenv import load_dotenv

# 기본 학습은 내 컴퓨터에서 실제 VLM으로 추론합니다.
USE_OPENAI = False
if "local_server" in globals() and local_server is not None:
    local_server.shutdown()
    local_server.server_close()
local_server = None

if USE_OPENAI:
    load_dotenv(Path(".env"))
    MODEL = "gpt-4.1-mini"
    client = OpenAI(max_retries=0,timeout=120.0)
else:
    import torch
    from transformers import AutoProcessor, AutoModelForImageTextToText
    torch.set_num_threads(2)
    MODEL = "HuggingFaceTB/SmolVLM-256M-Instruct"
    revision = "d039ab0d4b9311b617e8ffec1d06c8464364aeb3"
    processor = AutoProcessor.from_pretrained(MODEL,revision=revision)
    processor.image_processor.do_image_splitting = False
    processor.image_processor.size = {"longest_edge":512}
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL,revision=revision,torch_dtype=torch.float32,attn_implementation="eager").eval()

    @torch.inference_mode()
    def infer(raw,question):
        chat = [{"role":"user","content":[{"type":"image"},{"type":"text","text":question}]}]
        text = processor.apply_chat_template(chat,tokenize=False,add_generation_prompt=True)
        inputs = processor(text=text,images=[rgb(raw)],return_tensors="pt")
        tokens = model.generate(**inputs,max_new_tokens=80,do_sample=False)
        new = tokens[:,inputs.input_ids.shape[1]:]
        answer = processor.batch_decode(new,skip_special_tokens=True)[0].strip()
        eos = model.generation_config.eos_token_id
        stops = eos if isinstance(eos,list) else [eos]
        limited = new.shape[1]==80 and int(new[0,-1]) not in stops
        return answer,limited

    # 이 수업에서 쓰는 텍스트 1개 + 이미지 1개의 Responses 요청만 처리합니다.
    class LocalVLM(BaseHTTPRequestHandler):
        def log_message(self,*args):
            pass
        def send_json(self,status,data):
            body = json.dumps(data,ensure_ascii=False).encode("utf-8")
            self.send_response(status)
            self.send_header("Content-Type","application/json")
            self.send_header("Content-Length",str(len(body)))
            self.end_headers()
            self.wfile.write(body)
        def do_POST(self):
            if self.path!="/v1/responses":
                self.send_json(404,{"error":{"message":"지원하지 않는 경로"}})
                return
            try:
                body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
                parts = body["input"][0]["content"]
                question = next(x["text"] for x in parts if x["type"]=="input_text")
                url = next(x["image_url"] for x in parts if x["type"]=="input_image")
                prefix,encoded = url.split(",",1)
                if prefix!="data:image/jpeg;base64":
                    raise ValueError("JPEG 데이터 URL을 사용하세요.")
                raw = base64.b64decode(encoded,validate=True)
                fmt = body.get("text",{}).get("format",{})
                if fmt.get("type")=="json_schema":
                    question += "\nRequested JSON schema: "+json.dumps(fmt["schema"])
                answer,limited = infer(raw,question)
            except (ValueError,KeyError,StopIteration,OSError) as exc:
                self.send_json(400,{"error":{"message":str(exc)}})
                return
            except Exception as exc:
                self.send_json(500,{"error":{"message":type(exc).__name__}})
                return
            suffix = uuid.uuid4().hex
            status = "incomplete" if limited else "completed"
            self.send_json(200,{"id":"resp_"+suffix,"object":"response","created_at":int(time.time()),
                "status":status,"incomplete_details":{"reason":"max_output_tokens"} if limited else None,
                "model":MODEL,"parallel_tool_calls":False,"tools":[],"tool_choice":"auto",
                "output":[{"type":"message","id":"msg_"+suffix,"role":"assistant","status":status,
                "content":[{"type":"output_text","text":answer,"annotations":[]}]}]})
    local_server = HTTPServer(("127.0.0.1",0),LocalVLM)
    threading.Thread(target=local_server.serve_forever,daemon=True).start()
    port = local_server.server_address[1]
    client = OpenAI(base_url=f"http://127.0.0.1:{port}/v1",api_key="local-only",
                    max_retries=0,http_client=httpx.Client(trust_env=False,timeout=120))
print("요청 대상", MODEL, "|", "OpenAI 원격 API" if USE_OPENAI else "로컬 HTTP VLM 서버")

ModuleNotFoundError: No module named 'httpx'

### 공통 준비 ③ · 결과를 보존하는 추출 함수

| 필드 | 값과 판독 기준 |
|---|---|
| product_type | 물건 종류를 나타내는 문자열 또는 null |
| main_color | 실제 덮개 색을 확인한 경우에만 색 이름. 흑백이라 알 수 없으면 null |
| visible_features | 확인할 수 있는 외형 특징을 문자열 목록으로 기록. 없으면 빈 목록 |
| scene | 촬영 상황을 나타내는 문자열 또는 null. 실습에서는 실내·야외를 비교 |

`make_input`은 SDK 메시지를 만듭니다. 필수 1에서 직접 같은 메시지를 구성해 검산합니다. `ProductInfo`는 네 키를 필수로 두고 키별 자료형을 검사합니다. 구조를 통과한 값은 사진으로 다시 확인합니다.

`extract`는 SDK의 `responses.with_raw_response.parse`를 사용해 원문과 파싱 결과를 함께 다룹니다. 정상적인 객체를 읽으면 그대로 반환합니다. 토큰 한도 또는 구조 오류일 때는 네 항목을 각각 다시 묻고, 실제 응답의 앞뒤 공백과 끝 마침표를 정리해 같은 스키마로 검사합니다. 특징 한 문구는 한 원소짜리 목록으로 넣습니다. 거절이나 빈 응답은 별도 상태로 남깁니다. `guarded_extract`는 연결·HTTP 예외도 사진별 상태로 바꾸므로 한 사진이 실패해도 다음 사진을 처리할 수 있습니다. 응답 원문, 처리 경로, 검사용 객체는 각각 보관합니다.

In [ ]:
def make_input(raw,question):
    encoded = base64.b64encode(raw).decode("ascii")
    return [{"role":"user","content":[{"type":"input_text","text":question},
            {"type":"input_image","image_url":"data:image/jpeg;base64,"+encoded}]}]
def ask(raw,question):
    resp = client.responses.create(model=MODEL,input=make_input(raw,question))
    return resp.status,resp.output_text
class ProductInfo(BaseModel):
    model_config = ConfigDict(extra="forbid",strict=True)
    product_type: str | None
    main_color: str | None
    visible_features: list[str]
    scene: str | None

structured_q = ("Extract the main umbrella's product_type, main_color, visible_features and scene. "
                "Use English. If the real canopy color cannot be determined, use null. "
                "Return only one JSON object. All four keys are required.")
questions = {
    "product_type":"Name the object with a canopy in this picture. Answer with one noun.",
    "main_color":"What color is the main umbrella canopy? Reply with one color word. If this is a black and white photograph reply UNKNOWN.",
    "visible_features":"Describe one visible feature of the main umbrella in a short phrase.",
    "scene":"Was this picture taken indoors or outdoors? Answer with one word."
}
def clean(raw):
    text = raw.strip().rstrip(".").strip()
    return None if text.lower() in {"","unknown","unclear","null","undetermined"} else text
def extract(key):
    raw = jpeg(key)
    reply = client.responses.with_raw_response.parse(
        model=MODEL,input=make_input(raw,structured_q),text_format=ProductInfo)
    envelope = reply.http_response.json()
    message_parts = [p for m in envelope.get("output",[]) for p in m.get("content",[])]
    original = "\n".join(p["text"] for p in message_parts if p["type"]=="output_text")
    status = envelope.get("status")
    log = {"image_id":key,"initial_status":status,"initial_raw":original,"field_raw":{}}
    reason = (envelope.get("incomplete_details") or {}).get("reason")
    if any(p["type"]=="refusal" for p in message_parts):
        return {**log,"route":"거절","data":None}
    if status=="incomplete" and reason=="max_output_tokens":
        log["parse_error"] = "max_output_tokens"
    elif status!="completed" or not original.strip():
        return {**log,"route":"응답 없음","data":None}
    else:
        try:
            parsed = reply.parse().output_parsed
        except ValidationError as exc:
            log["parse_error"] = exc.errors()[0]["type"]
        else:
            if parsed is None:
                return {**log,"route":"응답 없음","data":None}
            return {**log,"route":"직접 파싱","data":parsed.model_dump()}
    values = {}
    for field,prompt in questions.items():
        status,answer = ask(raw,prompt)
        log["field_raw"][field] = answer
        if status!="completed" or not answer.strip():
            return {**log,"route":"응답 없음","data":None}
        value = clean(answer)
        values[field] = ([value] if value else []) if field=="visible_features" else value
    checked = ProductInfo.model_validate(values)
    return {**log,"route":"필드별 재추론","data":checked.model_dump()}

def guarded_extract(key):
    try:
        return extract(key)
    except openai.APIConnectionError as exc:
        return {"image_id":key,"route":"연결 실패","data":None,"error":type(exc).__name__}
    except openai.APIStatusError as exc:
        return {"image_id":key,"route":"요청 실패","data":None,"http_status":exc.status_code}

## 필수 1 · 이미지와 질문을 한 요청에 담기

사진을 읽는 일, 전송용 문자열을 만드는 일, 모델이 그 사진을 보고 답하는 일을 분리해 확인하세요.

### 1-1. 네 사진에서 실제 색을 판독할 수 있는지 조사하기

네 원본의 포맷·모드·해상도·파일 크기·최대 채널 차이를 표로 만들고 사진을 표시하세요. 9448과 40083에서 우산의 실제 색을 기록할 수 있는 근거를 각각 쓰세요.

**풀이 단서**  
파일의 원래 모드는 Image.open에서 읽고, 채널 차이는 공통 함수 chroma에서 구합니다. 순서는 ids 목록을 따르세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# overview = []
# for key in ids: ...
# 각 행을 표로 만든 뒤 show로 사진을 함께 확인하세요.

**나의 답**  
실행값 또는 사진에서 확인한 근거를 인용해 설명하세요.

✎

### 1-2. Base64 이미지 요청 만들고 바이트 단위로 복원하기

9448 원본을 Base64로 인코딩해 질문과 함께 user 메시지 하나에 넣으세요. 접두어와 두 항목의 type, 원본·문자열 길이, 디코딩 결과를 출력하세요. 직접 만든 요청과 공통 함수 결과가 같은지 검산하세요.

**풀이 단서**  
Base64 문자열 길이는 4×ceil(원본 바이트 수/3)입니다. 출력에는 데이터 URL 전체 대신 쉼표 앞의 접두어만 표시하세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# question = "Describe the main umbrella's visible color and features in one sentence."
# request_input = ...
# 디코딩한 bytes를 jpeg(9448)과 비교하세요.

**나의 답**  
실행값 또는 사진에서 확인한 근거를 인용해 설명하세요.

✎

### 1-3. 완료 상태와 답의 신뢰성 구별하기

같은 질문으로 9448과 40083에 자연어 요청을 보내세요. status와 output_text를 원문으로 출력하고 색과 외형에 관한 주장을 사진별로 검수하세요. 두 요청이 모두 완료됐을 때도 어떤 정보를 보류할 수 있는지 설명하세요.

**풀이 단서**  
첫 번째 요청은 client.responses.create로 직접 실행하고, 둘째 요청은 같은 일을 하는 ask를 써도 됩니다. 원본 소개 문장과 사진을 한 화면에서 읽으세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# response = client.responses.create(model=MODEL, input=request_input)
# 40083에도 같은 question을 적용하세요.

**나의 답**  
실행값 또는 사진에서 확인한 근거를 인용해 설명하세요.

✎

### 1-4. 같은 우산을 흑백으로 바꿔 색상 보류 시험하기

컬러 원본 3장과 각 흑백 변환본에 동일한 색상 질문을 보내세요. 파일 ID·입력 조건·최대 채널 차이·원문·기대값·통과 여부를 남기고, 원본 색상 일치율과 흑백 보류율을 각각 계산하세요.

**풀이 단서**  
원본의 참조 덮개 색은 9448 blue, 29984 green, 84477 blue입니다. 흑백 입력의 목표는 원본 색을 맞히는 것보다 보류하는 것입니다. 분모는 조건마다 3입니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# pairs = []
# for key, truth in ...: 원본과 mono_jpeg 원본을 각각 ask에 전달하세요.
# 질문은 questions["main_color"]를 고정하세요.

**나의 답**  
실행값 또는 사진에서 확인한 근거를 인용해 설명하세요.

✎

## 필수 2 · 한 문장을 네 필드의 표로 바꾸기

자료형을 먼저 정하고 한 장의 요청 기록을 따라간 뒤 네 장으로 넓힙니다. 마지막에는 “표를 만들었는가?”와 “표의 내용이 맞는가?”를 다른 기준으로 집계합니다.

### 2-1. null을 허용하는 필수 키와 누락된 키 구별하기

ProductInfo의 네 필드를 확인하세요. 색이 null인 사례에서 시작해 색 키 삭제, 특징 목록을 단일 문자열로 변경, 미등록 price 키 추가를 하나씩 시험하세요. 실패한 필드와 오류 유형을 출력하세요.

**풀이 단서**  
한 번에 하나씩만 바꿉니다. model_validate는 파이썬 딕셔너리를 검사하고 ValidationError.errors()의 loc와 type이 문제 위치와 유형을 알려 줍니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# template = {"product_type":"umbrella","main_color":None,"visible_features":[],"scene":"outdoors"}
# 각 변형을 try/except로 검사하세요.

**나의 답**  
실행값 또는 사진에서 확인한 근거를 인용해 설명하세요.

✎

### 2-2. 9448의 구조화 요청을 추적하고 원문 보존하기

guarded_extract로 9448 한 건만 요청해 cache에 보관하세요. 첫 상태·응답 원문·파싱 진단·최종 처리 경로·최종 객체를 출력하세요. 첫 응답이 실패했는데도 최종 객체가 생길 수 있는 과정을 설명하세요.

**풀이 단서**  
initial_raw와 field_raw를 따로 보세요. 이미 성공한 결과는 다음 문항에서 다시 요청하지 않습니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# cache = {9448:guarded_extract(9448)}
# 같은 항목의 initial_status, initial_raw, parse_error, route, data를 살펴보세요.

**나의 답**  
실행값 또는 사진에서 확인한 근거를 인용해 설명하세요.

✎

### 2-3. 기존 결과를 재사용해 배치 표 작성하기

9448은 재사용하고 다른 세 장을 추가해 사진당 한 행인 표를 만드세요. ID·처리 모델·처리 경로·첫 상태·진단·네 필드를 기록하세요. 최종 객체가 없는 사진도 행을 유지하고 처리된 사진 수와 객체 수를 나눠 보고하세요.

**풀이 단서**  
캐시에 ID가 없을 때만 요청합니다. data가 None인 행에도 ID와 실패 상태는 남겨야 전체 분모가 보존됩니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# for key in ids: ...
# rows에 사진별 처리 경로와 실제 필드를 붙여 DataFrame으로 만드세요.

**나의 답**  
실행값 또는 사진에서 확인한 근거를 인용해 설명하세요.

✎

### 2-4. 구조 복구율과 실제 필드 일치율을 따로 집계하기

네 사진에서 우산 개별 주석 수와 사람의 색 판독값을 확인하세요. 객체 수신·종류 일치·색 일치·두 항목 동시 일치를 각기 세고 분모를 제시하세요. 모델이 배경을 종류로 기록한 사례도 찾으세요.

**풀이 단서**  
product_type과 main_color를 각각 비교한 다음 두 조건의 논리곱을 계산합니다. 단어 대소문자 차이는 정리하되 새 색이나 품종을 채우지 마세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# references = {9448:"blue",40083:None,29984:"green",84477:"blue"}
# 이미지별 참·거짓 열을 만든 뒤 합계를 계산하세요.

**나의 답**  
실행값 또는 사진에서 확인한 근거를 인용해 설명하세요.

✎

## 과제 1 · 요청 실패, 구조 오류, 내용 오류를 나누어 대응하기

사진 40083으로 독립적으로 진행할 수 있습니다. 먼저 사람이 통제한 사례 A~D로 검수 규칙을 확인하고, 마지막 문항에서는 같은 기준을 실제 모델의 응답에 적용합니다. 사례 A~D의 건수는 실제 모델 호출의 실패율에 포함하지 않습니다.

### 3-1. 흑백 사진의 판독 근거와 A~D 사례 구성하기

사진 40083과 COCO 우산 주석을 확인하고 네 필드의 참조 딕셔너리를 작성하세요. A는 참조와 같게, B는 색만 red로, C는 색 키를 삭제해, D는 전체 응답이 없도록 구성하세요. 어느 필드를 사람이 판독했는지 적으세요.

**풀이 단서**  
사진에서 읽히는 외형은 펼쳐진 덮개와 글자로 정합니다. 실제 색은 null입니다. Python에서는 객체 전체가 없는 D도 None으로 나타내지만 A의 main_color와 자리 자체가 다릅니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# base_ref의 특징은 ["open canopy","lettering"], 장소는 "outdoors"로 맞춰 보세요.
# cases["A"]부터 cases["D"]까지 만들고 근거를 기록하세요.

**나의 답**  
실행값 또는 사진에서 확인한 근거를 인용해 설명하세요.

✎

### 3-2. 검수 순서를 구현하고 네 가지 상태 판정하기

전체 객체 유무 → 스키마 → 항목별 내용 순으로 검사하는 review를 작성하세요. A~D의 상태와 문제 필드를 출력하고, 특징 목록의 순서만 바꾸었을 때 결과가 유지되는지 대조하세요.

**풀이 단서**  
특징의 순서와 중복을 이 과제에서는 채점하지 않습니다. 문자열을 정규화하고 집합으로 비교하세요. 표현이 다른 자유문장의 최종 판정은 사진을 보며 합니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# def review(payload, reference): ...
# 같은 자료를 동일한 함수로 검사하세요.

**나의 답**  
실행값 또는 사진에서 확인한 근거를 인용해 설명하세요.

✎

### 3-3. 실제 요청 오류 진단과 내용 정정 구별하기

로컬 모드에서 Base64가 깨진 요청을 한 번 보내 HTTP 상태와 조치를 확인하세요. 401·429·503·연결 끊김의 대응도 정리하세요. 이어 B의 색상만 수정한 사본을 만들어 같은 검수 함수로 전후를 비교하세요.

**풀이 단서**  
로컬 실험은 의도적으로 @@@을 넣어 400을 일으킵니다. 원격 모드에서는 이 호출을 건너뜁니다. 잘못된 바이트는 요청 본문을 수정해야 하고 사진 내용 오류는 필드 근거를 확인해야 합니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# next_action을 작성하고 openai.APIStatusError에서 status_code를 확인하세요.
# repaired는 B의 원본을 덮어쓰지 않는 사본입니다.

**나의 답**  
실행값 또는 사진에서 확인한 근거를 인용해 설명하세요.

✎

### 3-4. 실제 모델 결과를 정정하고 남은 문제까지 보고하기

실제 40083 결과를 가져와 색 정보가 없는 입력에서 main_color만 null로 바꾸는 규칙을 사본에 적용하세요. 같은 review로 전후의 문제 필드를 비교하고, 자동 등록해도 되는지 또는 어떤 내용을 사람이 더 확인해야 하는지 결정하세요.

**풀이 단서**  
필수 2를 풀었다면 캐시를 재사용하고, 과제만 진행했다면 guarded_extract로 한 번 새로 받으세요. 실제 모델이 적은 특징 문장에도 색상 주장이 반복될 수 있습니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# before는 실제 모델 결과입니다.
# after는 사본으로 만들고 두 결과를 base_ref와 각각 비교하세요.

**나의 답**  
실행값 또는 사진에서 확인한 근거를 인용해 설명하세요.

✎

## 실습 마무리

1. **어떤 문제가 있었나요?** 응답 완료·스키마 통과·사진 근거 일치가 서로 다른 결과를 낸 실제 사례를 쓰세요.
2. **어떻게 개선했나요?** 프롬프트로 보류를 요청한 결과와 모델 응답 이후 사람이 정한 보류 규칙을 적용한 결과를 구분해 쓰세요.
3. **무엇으로 판단했나요?** 사진 ID, 원문, 구조화 경로, 항목별 수치 중 세 가지 이상을 사용해 등록 판단과 남은 검토 사항을 설명하세요.